# Şoför Risk Skoru — Ham Veriden İnşa

**Kaynak:** `iett_data.db` → ariza, kaza, denetim, sefer tabloları (H1 2025)  
**Çıktı:** `panel_data/sofor_risk.json` + `iett_panel/panel_data/sofor_risk.json`

## 6 Metrik, Toplam Ağırlık = 1.0

| Metrik | Kaynak | Ağırlık | Gerekçe |
|---|---|---|---|
| Ciddi kaza (can kaybı) | kaza: OLUM+YARALI > 0 | **0.30** | En kritik güvenlik riski |
| Tüm kaza oranı | kaza / sefer × 1000 | 0.22 | Genel kaza eğilimi |
| Şoför kusurlu kaza | KUSURGRUBU = Sürücü Kusurları | 0.18 | Doğrudan sorumluluk |
| Denetim olumsuz oranı | CEVAPSECENEKDURUM = 0 | 0.15 | Uyumsuzluk |
| Tekrarlayan arıza (2 ayda 4+) | Son 60g, aynı kategoride 4+ | 0.10 | Sürekli arıza yaratma |
| Yaya kazası | OLUMYAYA+YARALIYAYA > 0 | 0.05 | Yaya güvenliği |

**Dışlanan:** Yarıda kalan sefer (araç arızası, şoför sorumluluğu değil)

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import json
import shutil
from pathlib import Path
from datetime import timedelta

DATATHON_DIR = Path(r"c:\Users\asus\Desktop\Datathon")
PANEL_DIR    = DATATHON_DIR / "panel_data"
DB_PATH      = PANEL_DIR / "iett_data.db"

MIN_SEFER = 10
KUSUR_MULTIPLIER = 0.15   # Kusurlu kaza %15 daha riskli (çift sayma yok)

# V4: Şoför kusur kazanın multiplier"ı; yaya 0.05→0.15
AGIRLIKLAR = {
    "ciddi_kaza":   0.30,
    "kaza_etkili":  0.22,   # kaza_sayisi + kusur * 0.15 (V3 0.22 korundu)
    "yaya":         0.15,
    "denetim":      0.20,
    "tekrar":       0.13,
}
assert abs(sum(AGIRLIKLAR.values()) - 1.0) < 0.001

KATEGORI_ESIK = [
    (80, "KRİTİK",  "Kritik Risk – Acil Müdahale",   "#dc2626"),
    (60, "RİSK",    "Risk Grubunda – Yakın Takip",    "#f97316"),
    (40, "İYİ",     "Kabul Edilebilir Performans",    "#3b82f6"),
    (20, "GELİŞİM", "Gelişim Potansiyeli",            "#a855f7"),
    ( 0, "ÜSTÜN",   "Üstün Performans – Ödül Adayı", "#22c55e"),
]

ODUL_SISTEMI = {
    "ÜSTÜN":   {"label": "Üstün Performans",   "renge": "#22c55e",
                "oduller": ["Aylık sefer süresi azaltma hakkı","Tercihli hat ataması","Yıllık performans primi"]},
    "GELİŞİM": {"label": "Gelişim Programı",   "renge": "#a855f7",
                "oduller": ["Gönüllü eğitim desteği","Mentorluk programı"]},
    "İYİ":     {"label": "Standart Performans","renge": "#3b82f6",
                "oduller": ["Rutin değerlendirme","Standart hat ataması"]},
    "RİSK":    {"label": "Risk Grubunda",      "renge": "#f97316",
                "oduller": ["Zorunlu defansif sürüş eğitimi","Artırılmış denetim","Yönetici bilgilendirmesi"]},
    "KRİTİK":  {"label": "Kritik Risk – Acil Müdahale","renge": "#dc2626",
                "oduller": ["Acil müdahale: hat değişimi","Yönetici görüşmesi","Detaylı değerlendirme"]},
}

conn = sqlite3.connect(DB_PATH)
print(f"DB: {DB_PATH}")
for t in ["ariza","kaza","denetim","sefer"]:
    n = pd.read_sql(f"SELECT COUNT(*) n FROM {t}", conn)["n"][0]
    print(f"  {t}: {n:,} satir")


DB: c:\Users\asus\Desktop\Datathon\panel_data\iett_data.db
  ariza: 78,300 satir
  kaza: 3,039 satir
  denetim: 2,869,509 satir
  sefer: 10,162,086 satir


In [2]:
# ── 1. SEFER: temel metrikler ─────────────────────────────────────────────────
print('1. Sefer...')
sefer_df = pd.read_sql("""
    SELECT SICILNO, KAPINO, HATKODU
    FROM sefer
    WHERE SICILNO IS NOT NULL AND SICILNO != ''
""", conn)

sefer_agg = sefer_df.groupby('SICILNO').agg(
    sefer_sayisi    = ('HATKODU', 'count'),
    hat_cesitliligi = ('HATKODU', 'nunique'),
    kapino_cesit    = ('KAPINO',  'nunique'),
).reset_index()

sefer_agg = sefer_agg[sefer_agg['sefer_sayisi'] >= MIN_SEFER].copy()
print(f'  {len(sefer_agg):,} şoför (min {MIN_SEFER} sefer)')

1. Sefer...
  13,871 şoför (min 10 sefer)


In [3]:
# ── 2. KAZA: genel + ciddi (can kaybı) + şoför kusuru + yaya ──────────────────
print('2. Kaza...')
kaza_df = pd.read_sql("""
    SELECT
        SICILNO,
        OLUMYAYA, YARALIYAYA,
        OLUMYOLCU, YARALIYOLCU,
        OLUMSURUCU, YARALISURUCU,
        OLUMSIVIL, YARALISIVIL,
        KUSURGRUBU
    FROM kaza
    WHERE SICILNO IS NOT NULL AND SICILNO != ''
""", conn)

# Can kaybı olan kaza (ölüm veya yaralı — tüm taraflar)
can_kolonlari = ['OLUMYAYA','YARALIYAYA','OLUMYOLCU','YARALIYOLCU',
                 'OLUMSURUCU','YARALISURUCU','OLUMSIVIL','YARALISIVIL']
kaza_df['ciddi_kaza_flag'] = kaza_df[can_kolonlari].sum(axis=1).gt(0).astype(int)

# Yaya kazası
kaza_df['yaya_flag'] = (kaza_df['OLUMYAYA'] + kaza_df['YARALIYAYA']).gt(0).astype(int)

# Şoför kusurlu kaza — KUSURGRUBU'nda encoding sorunu var, LIKE pattern kullan
kaza_df['sofor_kusur_flag'] = kaza_df['KUSURGRUBU'].str.contains(
    r'r.c.*Kusur|Kusur.*r.c', na=False, regex=True, case=False
).astype(int)

kaza_agg = kaza_df.groupby('SICILNO').agg(
    kaza_sayisi      = ('SICILNO',         'count'),
    ciddi_kaza_n     = ('ciddi_kaza_flag',  'sum'),
    yaya_kaza_n      = ('yaya_flag',        'sum'),
    sofor_kusurlu_n  = ('sofor_kusur_flag', 'sum'),
).reset_index()

print(f'  {len(kaza_agg):,} şoför | Kaza: {kaza_agg["kaza_sayisi"].sum():,}')
print(f'  Ciddi (can kaybı): {kaza_agg["ciddi_kaza_n"].sum()} | '
      f'Yaya: {kaza_agg["yaya_kaza_n"].sum()} | '
      f'Şoför kusuru: {kaza_agg["sofor_kusurlu_n"].sum()}')

2. Kaza...
  2,385 şoför | Kaza: 3,039
  Ciddi (can kaybı): 78 | Yaya: 28 | Şoför kusuru: 232


In [4]:
# ── 3. TEKRARLAYAN ARIZA: son 60 gün içinde aynı kategoride 4+ ────────────────
# (Sınıflandırılmadı ve Destek kategorileri hariç — şoföre atfedilemez)
print('3. Tekrarlayan arıza (son 60g, 4+ aynı kategori)...')

ariza_df = pd.read_sql("""
    SELECT SOFOR_SICILNO, ARIZAUSTKODTANIM, OLAYTARIHI
    FROM ariza
    WHERE SOFOR_SICILNO IS NOT NULL AND SOFOR_SICILNO != ''
      AND SOFOR_MESLEK LIKE '%of%r'
      AND ARIZAUSTKODTANIM NOT LIKE '%n%fland%r%lmad%'
      AND ARIZAUSTKODTANIM NOT LIKE 'Destek'
      AND ARIZAUSTKODTANIM NOT LIKE 'Genel%'
""", conn)

ariza_df['OLAYTARIHI'] = pd.to_datetime(ariza_df['OLAYTARIHI'])

REF_DATE   = ariza_df['OLAYTARIHI'].max()   # H1 sonu = 2025-06-30
PENCERE_60 = REF_DATE - timedelta(days=60)  # 2025-05-01 civarı
print(f'  Referans: {REF_DATE.date()} | 60g pencere: {PENCERE_60.date()}')

# Son 60 gün penceresi
son60 = ariza_df[ariza_df['OLAYTARIHI'] >= PENCERE_60].copy()

# Şoför × kategori bazında sayı — 4+ olanlar "tekrarlayan"
tekrar_base = (
    son60.groupby(['SOFOR_SICILNO', 'ARIZAUSTKODTANIM'])
         .size()
         .reset_index(name='adet')
)
tekrar_base = tekrar_base[tekrar_base['adet'] >= 4]

# Şoför başına: kaç farklı kategoride tekrarlayan arıza var?
tekrar_agg = (
    tekrar_base.groupby('SOFOR_SICILNO')
               .size()
               .reset_index(name='tekrar_kategori_n')
)
tekrar_agg.columns = ['SICILNO', 'tekrar_kategori_n']

print(f'  Tekrarlayan arızası olan şoför: {len(tekrar_agg):,}')
print(f'  Dağılım:\n{tekrar_base["ARIZAUSTKODTANIM"].value_counts().head(8).to_string()}')

3. Tekrarlayan arıza (son 60g, 4+ aynı kategori)...
  Referans: 2025-06-30 | 60g pencere: 2025-05-01
  Tekrarlayan arızası olan şoför: 256
  Dağılım:
ARIZAUSTKODTANIM
KLİMA SİSTEMİ ARIZALARI          87
MOTOR ARIZALARI                  50
SOĞUTMA SİSTEMİ ARIZASI          36
KAPI ARIZALARI                   31
SÜSPANSİYON SİSTEMİ ARIZALARI    22
YAKIT ve ENJEKSİYON ARIZALARI    18
ELEKTRİK SİSTEMİ ARIZALARI       12
OTOMATİK ŞANZIMAN ARIZALARI       7


In [5]:
# ── 4. DENETİM: olumsuz oranı ─────────────────────────────────────────────────
print('4. Denetim...')
denetim_df = pd.read_sql("""
    SELECT SOFOR_SICILNO, CEVAPSECENEKDURUM
    FROM denetim
    WHERE SOFOR_SICILNO IS NOT NULL AND SOFOR_SICILNO != ''
      AND SOFOR_MESLEK LIKE '%of%r'
""", conn)

denetim_agg = denetim_df.groupby('SOFOR_SICILNO').agg(
    denetim_sayisi  = ('CEVAPSECENEKDURUM', 'count'),
    denetim_olumsuz = ('CEVAPSECENEKDURUM', lambda x: (x == 0).sum()),
).reset_index()
denetim_agg.columns = ['SICILNO', 'denetim_sayisi', 'denetim_olumsuz']
denetim_agg['denetim_oran_pct'] = (
    denetim_agg['denetim_olumsuz'] / denetim_agg['denetim_sayisi'] * 100
).round(1)

print(f'  {len(denetim_agg):,} şoför')

4. Denetim...
  12,856 şoför


In [6]:
# ── 5. BİRLEŞTİRME ────────────────────────────────────────────────────────────
print('5. Birleştirme...')
df = sefer_agg.copy()
df = df.merge(kaza_agg,    on='SICILNO', how='left')
df = df.merge(tekrar_agg,  on='SICILNO', how='left')
df = df.merge(denetim_agg, on='SICILNO', how='left')

# Eksik → 0 (o şoförde o olay gerçekleşmemiş)
sifir_kolonlar = [
    'kaza_sayisi','ciddi_kaza_n','yaya_kaza_n','sofor_kusurlu_n',
    'tekrar_kategori_n',
    'denetim_sayisi','denetim_olumsuz','denetim_oran_pct',
]
for col in sifir_kolonlar:
    df[col] = df[col].fillna(0)

# Per-1000 sefer normalize
df['kaza_per1k']       = (df['kaza_sayisi']      / df['sefer_sayisi'] * 1000).round(3)
df['ciddi_kaza_per1k'] = (df['ciddi_kaza_n']     / df['sefer_sayisi'] * 1000).round(3)

print(f'  {len(df):,} şoför')
print(f'  Kaza olan: {(df["kaza_sayisi"]>0).sum():,} | '
      f'Ciddi kaza olan: {(df["ciddi_kaza_n"]>0).sum():,} | '
      f'Tekrarlayan arızalı: {(df["tekrar_kategori_n"]>0).sum():,}')

5. Birleştirme...
  13,871 şoför
  Kaza olan: 2,383 | Ciddi kaza olan: 38 | Tekrarlayan arızalı: 256


In [7]:
# --- 6. RİSK SKORU (V4: kusur multiplier, yaya yükseltildi) ---
def prank(s):
    return s.rank(pct=True, method="average").mul(100).round(1)

# Kaza etkili: kusurlu kaza ×1.15 (çift sayma yok)
df["kaza_skoru_etkili"]   = df["kaza_sayisi"] + df["sofor_kusurlu_n"] * KUSUR_MULTIPLIER
df["kaza_per1k"]          = (df["kaza_sayisi"]       / df["sefer_sayisi"] * 1000).round(3)
df["kaza_per1k_etkili"]   = (df["kaza_skoru_etkili"] / df["sefer_sayisi"] * 1000).round(3)
df["ciddi_kaza_per1k"]    = (df["ciddi_kaza_n"]      / df["sefer_sayisi"] * 1000).round(3)

df["z_ciddi_kaza"]  = prank(df["ciddi_kaza_per1k"])
df["z_kaza_etkili"] = prank(df["kaza_per1k_etkili"])
df["z_yaya"]        = prank(df["yaya_kaza_n"])
df["z_denetim"]     = prank(df["denetim_oran_pct"])
df["z_tekrar"]      = prank(df["tekrar_kategori_n"])

df["risk_skoru"] = (
    df["z_ciddi_kaza"]  * AGIRLIKLAR["ciddi_kaza"]  +
    df["z_kaza_etkili"] * AGIRLIKLAR["kaza_etkili"] +
    df["z_yaya"]        * AGIRLIKLAR["yaya"]        +
    df["z_denetim"]     * AGIRLIKLAR["denetim"]     +
    df["z_tekrar"]      * AGIRLIKLAR["tekrar"]
).clip(0, 100).round(1)

# Quantile tier
q90, q75, q55, q30 = df["risk_skoru"].quantile([0.90, 0.75, 0.55, 0.30]).values
print(f"Tier eşikleri: q30={q30:.1f} q55={q55:.1f} q75={q75:.1f} q90={q90:.1f}")

def _kat(s):
    if s >= q90: return "KRİTİK"
    if s >= q75: return "RİSK"
    if s >= q55: return "İYİ"
    if s >= q30: return "GELİŞİM"
    return "ÜSTÜN"

df["kategori"] = df["risk_skoru"].map(_kat)
print("Dağılım:", df["kategori"].value_counts().to_dict())


Tier eşikleri: q30=44.9 q55=50.7 q75=55.3 q90=60.0
Dağılım: {'ÜSTÜN': 4127, 'GELİŞİM': 3349, 'İYİ': 2909, 'RİSK': 2098, 'KRİTİK': 1388}


In [8]:
# --- 7. JSON SEMASI (5 bilesen) ---
kat_label = {e[1]: e[2] for e in KATEGORI_ESIK}
kat_renge = {e[1]: e[3] for e in KATEGORI_ESIK}

soforler = []
for _, r in df.iterrows():
    kat = r["kategori"]
    soforler.append({
        "sicilno":           r["SICILNO"],
        "risk_skoru":        float(r["risk_skoru"]),
        "kategori":          kat,
        "kategori_label":    kat_label[kat],
        "renge":             kat_renge[kat],
        "oduller":           ODUL_SISTEMI[kat]["oduller"],
        "sefer_sayisi":      int(r["sefer_sayisi"]),
        "hat_cesitliligi":   int(r["hat_cesitliligi"]),
        "kapino_cesit":      int(r["kapino_cesit"]),
        "kaza_sayisi":       int(r["kaza_sayisi"]),
        "kaza_per1k":        float(r["kaza_per1k"]),
        "kaza_per1k_etkili": float(r["kaza_per1k_etkili"]),
        "sofor_kusurlu_n":   int(r["sofor_kusurlu_n"]),
        "ciddi_kaza_n":      int(r["ciddi_kaza_n"]),
        "ciddi_kaza_per1k":  float(r["ciddi_kaza_per1k"]),
        "yaya_kaza_n":       int(r["yaya_kaza_n"]),
        "tekrar_kategori_n": int(r["tekrar_kategori_n"]),
        "denetim_sayisi":    int(r["denetim_sayisi"]),
        "denetim_olumsuz":   int(r["denetim_olumsuz"]),
        "denetim_oran_pct":  float(r["denetim_oran_pct"]),
        "z_ciddi_kaza":      float(r["z_ciddi_kaza"]),
        "z_kaza_etkili":     float(r["z_kaza_etkili"]),
        "z_yaya":            float(r["z_yaya"]),
        "z_denetim":         float(r["z_denetim"]),
        "z_tekrar":          float(r["z_tekrar"]),
    })

out = {
    "meta": {
        "versiyon": "4.0",
        "donem":    "2025-H1",
        "kaynak":   "Ham SQLite: ariza + kaza + denetim + sefer",
        "agirliklar": {
            "ciddi_kaza_can_kaybi":      AGIRLIKLAR["ciddi_kaza"],
            "genel_kaza_kusur_carpanli": AGIRLIKLAR["kaza_etkili"],
            "yaya_kazasi":               AGIRLIKLAR["yaya"],
            "denetim_olumsuz":           AGIRLIKLAR["denetim"],
            "tekrarlayan_ariza":         AGIRLIKLAR["tekrar"],
        },
        "kusur_multiplier":  KUSUR_MULTIPLIER,
        "kusur_formul":      "kaza_skoru = kaza_sayisi + sofor_kusurlu * 0.15 (kusurlu kaza 1.15x agirlik)",
        "normalizasyon":     "percentile_rank",
        "tier_yontemi":      "quantile q30/q55/q75/q90",
        "tier_esikler":      f"q30={q30:.1f} q55={q55:.1f} q75={q75:.1f} q90={q90:.1f}",
        "min_sefer":         MIN_SEFER,
        "tekrar_pencere":    "Son 60 gun, ayni kategoride 4+ ariza (Siniflandirilmadi/Destek haric)",
        "dislanan":          "Yarida kalan sefer (arac arizasi, sofor sorumlulugu degil)",
        "v5_bagimliligi":    "YOK",
        "v4_degisiklik":     "Sofor kusur kaza icine multiplier olarak entegre (cift sayma yok); yaya 0.05->0.15",
    },
    "ozet": {
        "toplam_sofor":         len(soforler),
        "kategori_dagilim":     df["kategori"].value_counts().to_dict(),
        "ort_risk":             round(float(df["risk_skoru"].mean()), 1),
        "kaza_toplam":          int(df["kaza_sayisi"].sum()),
        "ciddi_kaza_toplam":    int(df["ciddi_kaza_n"].sum()),
        "yaya_kaza_toplam":     int(df["yaya_kaza_n"].sum()),
        "sofor_kusurlu_toplam": int(df["sofor_kusurlu_n"].sum()),
        "tekrarlayan_toplam":   int((df["tekrar_kategori_n"] > 0).sum()),
    },
    "odul_sistemi": ODUL_SISTEMI,
    "soforler":     soforler,
}
print(f"JSON hazir: {len(soforler):,} sofor")


JSON hazir: 13,871 sofor


In [9]:
# --- 8. YAZIM ---
OUT_D = PANEL_DIR / "sofor_risk.json"
OUT_P = Path(r"c:\\Users\\asus\\Desktop\\iett_panel\\panel_data\\sofor_risk.json")

with open(OUT_D, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print(f"Datathon: {OUT_D} ({OUT_D.stat().st_size/1024:.0f} KB)")

if OUT_P.parent.exists():
    shutil.copy2(OUT_D, OUT_P)
    print(f"Panel:    {OUT_P}")

conn.close()
print("DB kapatildi.")


Datathon: c:\Users\asus\Desktop\Datathon\panel_data\sofor_risk.json (11462 KB)
Panel:    c:\Users\asus\Desktop\iett_panel\panel_data\sofor_risk.json
DB kapatildi.


In [10]:
# --- 9. DOGRULAMA ---
with open(OUT_D, encoding="utf-8") as f:
    chk = json.load(f)

print(f"Versiyon:           {chk['meta']['versiyon']}")
print(f"V5 bagimliligi:     {chk['meta']['v5_bagimliligi']}")
print(f"Toplam sofor:       {chk['ozet']['toplam_sofor']:,}")
print(f"Kategori dagilimi:  {chk['ozet']['kategori_dagilim']}")
print(f"Ortalama risk:      {chk['ozet']['ort_risk']}")
print(f"Kaza toplam:        {chk['ozet']['kaza_toplam']:,}")
print(f"Ciddi kaza:         {chk['ozet']['ciddi_kaza_toplam']}")
print(f"Yaya kaza:          {chk['ozet']['yaya_kaza_toplam']}")
print(f"Sofor kusurlu:      {chk['ozet']['sofor_kusurlu_toplam']}")
print(f"Tekrarlayan sofor:  {chk['ozet']['tekrarlayan_toplam']:,}")

top5 = sorted(chk["soforler"], key=lambda s: s["risk_skoru"], reverse=True)[:5]
print()
print("--- TOP 5 RISK ---")
for s in top5:
    print(f"  {s['sicilno']}: skor={s['risk_skoru']} | ciddi={s['ciddi_kaza_n']} kaza={s['kaza_sayisi']}(kusur={s['sofor_kusurlu_n']}) yaya={s['yaya_kaza_n']} tekrar={s['tekrar_kategori_n']}")


Versiyon:           4.0
V5 bagimliligi:     YOK
Toplam sofor:       13,871
Kategori dagilimi:  {'ÜSTÜN': 4127, 'GELİŞİM': 3349, 'İYİ': 2909, 'RİSK': 2098, 'KRİTİK': 1388}
Ortalama risk:      50.0
Kaza toplam:        3,029
Ciddi kaza:         78
Yaya kaza:          28
Sofor kusurlu:      232
Tekrarlayan sofor:  256

--- TOP 5 RISK ---
  P_55266: skor=91.6 | ciddi=2 kaza=2(kusur=0) yaya=2 tekrar=0
  P_9812: skor=91.3 | ciddi=3 kaza=3(kusur=1) yaya=3 tekrar=0
  P_35664: skor=90.1 | ciddi=3 kaza=3(kusur=2) yaya=3 tekrar=0
  P_61022: skor=88.5 | ciddi=2 kaza=2(kusur=0) yaya=2 tekrar=0
  P_59163: skor=88.1 | ciddi=4 kaza=4(kusur=2) yaya=4 tekrar=1
